# Markowitz Portfolio Optimization: Does It Survive Out-of-Sample?

## Why this project

I recently looked into portfolio theory/management and stumbled across Markowitz's mean-variance framework. After learning a bit of linear algebra, statistics, and some Lagrange multipliers from multivariable calculus, I decided why not make one from scratch? Essentially, Markowitz's framework is minimizing risk (variance) for a given expected return, using the covariance structure between assets. It's taught everywhere, but rarely stress-tested by the people implementing it. Even when it is implemented, people tend to use modules like `scipy.optimize` or `PyPortfolioOpt` instead of building it from the ground up using Lagrange multipliers. It also tests whether a portfolio that's "optimal" on historical data actually performs well going forward, and compares a fix (covariance shrinkage) against the naive version and a simple 1/N benchmark.

(This is intended to be a learning experience and is by no means a superior way of implementing Markowitz's mean-variance framework.)

## Features

- Closed-form mean-variance optimizer: solves for optimal weights directly from the Lagrangian system, not via a numerical solver (like the scipy module)
- Efficient frontier construction across a range of target returns (stocks outlined later)
- Tangency (max Sharpe) portfolio is cross-validated against a `scipy.optimize` solution for validation
- Ledoit-Wolf covariance shrinkage is to address estimation error in near-singular covariance matrices
- Out-of-sample backtest: naive Markowitz vs. shrinkage-adjusted Markowitz vs. equal-weight (1/N), evaluated on realized (not estimated) returns
- Test suite validates the closed-form solution against numerical methods and checking basic invariants (weights sum to 1, target return is hit)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src import data, stats, optimizer, shrinkage, plotting

%matplotlib inline

## Derivation

So I don't have to re-derive this every time I forget: the problem is minimize $\tfrac{1}{2} w^T \Sigma w$ subject to $w^T \mu = R$ (hit a target return) and $w^T \mathbf{1} = 1$ (fully invested). Setting up the Lagrangian and solving $\nabla_w L = 0$ gives

$$ w = \lambda_1 \Sigma^{-1}\mu + \lambda_2 \Sigma^{-1}\mathbf{1} $$

so every efficient portfolio is just a combination of two vectors, $\Sigma^{-1}\mu$ and $\Sigma^{-1}\mathbf{1}$. That's two-fund separation, and it's the reason a closed form exists at all. Solving the two constraints for $\lambda_1, \lambda_2$ and plugging back in is what `optimizer.efficient_portfolio()` does. Full derivation with all the algebra is in the `src/optimizer.py` header, this is just the short version.

## Key design decisions

- `np.linalg.solve` over `np.linalg.inv` because matrix inversion increases numerical error when the covariance matrix is near-singular. Therefore solving the linear system directly is more stable and accurate.
- Closed-form implementation validated against `scipy.optimize` (checking answers), not used as the primary solver, just as a check on the hand-derived math.
- Annualized daily returns (×252) is standard convention for comparing to typical annual return/risk figures.
- Ledoit-Wolf shrinkage over raw sample covariance is because the sample covariance matrix is a poor estimator when the number of assets approaches the number of observations; shrinkage blends it toward a more stable, lower-variance target.

In [ ]:
TICKERS = ["AAPL", "MSFT", "AMZN", "GOOGL", "JPM", "XOM", "JNJ", "PG"]
START, END = "2015-01-01", "2024-01-01"
RF = 0.02  # annualized risk-free rate assumption

returns = data.get_returns(TICKERS, START, END)
mu, Sigma = stats.mu_sigma(returns)

print("Annualized mu:")
for t, m in zip(TICKERS, mu):
    print(f"  {t}: {m:.2%}")

In [ ]:
plotting.plot_covariance_heatmap(Sigma, TICKERS)
plt.show()

## Efficient frontier and tangency portfolio

Tangency (max Sharpe) portfolio is cross-validated against a `scipy.optimize` solution for validation, same thing `tests/test_optimizer.py` checks, just visible here so I can eyeball how close the two answers actually are.

In [ ]:
frontier = optimizer.efficient_frontier(mu, Sigma, n_points=60)
w_tan_closed = optimizer.tangency_portfolio(mu, Sigma, rf=RF)
w_tan_scipy = optimizer.tangency_portfolio_scipy(mu, Sigma, rf=RF)

print("Max |closed-form - scipy| weight difference:", np.max(np.abs(w_tan_closed - w_tan_scipy)))
print("Closed-form tangency Sharpe:", optimizer.portfolio_sharpe(w_tan_closed, mu, Sigma, rf=RF))

plotting.plot_efficient_frontier(frontier, mu, Sigma, tickers=TICKERS, tangency_weights=w_tan_closed, rf=RF)
plt.show()

## Does it survive out-of-sample?

This is the part that made the project worth doing: it also tests whether a portfolio that's "optimal" on historical data actually performs well going forward, and compares a fix (covariance shrinkage) against the naive version and a simple 1/N benchmark. Fit on the training window, weights held fixed, then evaluated on realized (not estimated) returns in the untouched test window. Three portfolios: naive Markowitz (raw sample covariance), shrinkage Markowitz (Ledoit-Wolf), and equal-weight (1/N, no estimation at all).

In [ ]:
result = shrinkage.train_test_backtest(returns, train_frac=0.6, rf=RF)

print(f"Naive Markowitz out-of-sample Sharpe: {result.naive_sharpe:.2f}")
print(f"Shrinkage-adjusted Markowitz out-of-sample Sharpe: {result.shrinkage_sharpe:.2f}")
print(f"Equal-weight (1/N) out-of-sample Sharpe: {result.equal_weight_sharpe:.2f}")

In [ ]:
plotting.plot_backtest_cumulative_returns(
    result.naive_realized_returns,
    result.shrinkage_realized_returns,
    result.equal_weight_realized_returns,
)
plt.show()

## Results (will be updated sometimes)

Backtest config: 8-asset universe (AAPL, MSFT, AMZN, GOOGL, JPM, XOM, JNJ, PG), 2015-01-01 to 2024-01-01 daily prices, 60% train / 40% test split, 2% annualized risk-free rate. Tangency (max-Sharpe) portfolio fit on the training window, weights held fixed and evaluated on realized test-window returns.

Naive Markowitz out-of-sample Sharpe: `-0.10`
Shrinkage-adjusted Markowitz out-of-sample Sharpe: `-0.08`
Equal-weight (1/N) out-of-sample Sharpe: `1.02`

Both Markowitz variants produced a negative out-of-sample Sharpe ratio, while the naive 1/N benchmark comfortably outperformed. This is an interesting thing to point out, however it is a well-documented result (DeMiguel, Garlappi & Uppal, 2009). The reason why is mean-variance weights are so sensitive to estimation error in μ that the "optimal" in-sample portfolio can be harmful out-of-sample. Another thing is that shrinkage helps at the margin here but doesn't flip the sign. As well, numbers will shift with the universe, date range, and train/test split (solution is to re-run `notebooks/analysis.ipynb` to reproduce or vary them).

## Limitations

Mean-variance optimization assumes stable, correctly-estimated inputs (μ, Σ). In practice they are noisy estimates from limited historical data. This project demonstrates that sensitivity directly rather than negating it (in the best interests of reducing margin of error).

## How to Run it

```bash
pip install -r requirements.txt
jupyter notebook notebooks/analysis.ipynb
```